# 06 — Scenario Simulator

Interactive 'what-if' simulator. Move the sliders to set daily inputs, see the predicted weight + body-fat trajectories with 95% credible bands.

Phase 1 caveat: parameters are drawn from population priors, so the bands will be wide. Phase 2 narrows them substantially by fitting `intake_bias` and `RMR_scale` to your data.

In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ipywidgets import interactive, FloatSlider, IntSlider, VBox, HBox

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from body_sim import simulate
from body_sim.config import DEFAULT_PROFILE
from body_sim.model import BodyState
from body_sim.validation import _seed_state

df = pd.read_parquet(REPO_ROOT / 'notebooks' / 'predictions' / 'daily_rollup.parquet')

# Seed state from the most recent observed weight
most_recent = df['weight_kg'].dropna().iloc[-1] if df['weight_kg'].notna().any() else 80.0
initial_state = _seed_state(float(most_recent))
print(f'Initial state: weight={initial_state.fat_mass_kg + initial_state.lean_mass_kg:.1f} kg, '
      f'fat={initial_state.fat_mass_kg:.1f} kg, lean={initial_state.lean_mass_kg:.1f} kg')

Initial state: weight=82.8 kg, fat=18.2 kg, lean=64.6 kg


In [2]:
def render(intake_kcal, protein_g, carb_g, fat_g, sodium_mg, steps, vigorous_min, horizon_days):
    inputs_per_day = [
        {
            'intake_kcal': intake_kcal,
            'protein_g': protein_g,
            'carb_g': carb_g,
            'fat_g': fat_g,
            'sodium_mg': sodium_mg,
            'ee_hr_keytel_kcal': 0.0,  # rely on fallback (steps) for scenario sim
            'workout_kcal': 0.0,
            'vigorous_min': vigorous_min,
            'intake_logged': True,
            'hr_coverage_pct': 0.0,
            'steps': steps,
        }
        for _ in range(horizon_days)
    ]
    samples = simulate.sample_parameters(n=200, seed=42)
    result = simulate.simulate_forward(
        initial_state=initial_state,
        inputs_per_day=inputs_per_day,
        profile=DEFAULT_PROFILE,
        parameter_samples=samples,
    )
    weight_band = simulate.credible_band(result.predicted_weight_kg)
    bf_band = simulate.credible_band(result.body_fat_pct)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    days = np.arange(1, horizon_days + 1)
    axes[0].fill_between(days, weight_band['lo'], weight_band['hi'], alpha=0.25, label='95% band')
    axes[0].plot(days, weight_band['median'], linewidth=2, label='Median')
    axes[0].set_xlabel('Day')
    axes[0].set_ylabel('Weight (kg)')
    axes[0].set_title(f'Weight after {horizon_days} days')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].fill_between(days, bf_band['lo'], bf_band['hi'], alpha=0.25)
    axes[1].plot(days, bf_band['median'], linewidth=2)
    axes[1].set_xlabel('Day')
    axes[1].set_ylabel('Body fat (%)')
    axes[1].set_title('Body fat trajectory')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'Predicted weight at day {horizon_days}: {weight_band["median"][-1]:.2f} kg '
          f'(95% CI: {weight_band["lo"][-1]:.2f}\u2013{weight_band["hi"][-1]:.2f})')
    print(f'Predicted body fat at day {horizon_days}: {bf_band["median"][-1]:.2f} % '
          f'(95% CI: {bf_band["lo"][-1]:.2f}\u2013{bf_band["hi"][-1]:.2f})')
    delta_w = weight_band['median'][-1] - weight_band['median'][0]
    print(f'Net weight change: {delta_w:+.2f} kg')

interactive(
    render,
    intake_kcal=FloatSlider(min=1200, max=3500, step=50, value=2000, description='kcal/day'),
    protein_g=FloatSlider(min=40, max=250, step=10, value=120, description='protein g'),
    carb_g=FloatSlider(min=50, max=400, step=10, value=200, description='carb g'),
    fat_g=FloatSlider(min=20, max=200, step=5, value=70, description='fat g'),
    sodium_mg=FloatSlider(min=1000, max=6000, step=100, value=2300, description='sodium mg'),
    steps=IntSlider(min=2000, max=20000, step=500, value=8000, description='steps'),
    vigorous_min=IntSlider(min=0, max=120, step=5, value=15, description='vigorous min'),
    horizon_days=IntSlider(min=7, max=180, step=7, value=56, description='horizon days'),
)

interactive(children=(FloatSlider(value=2000.0, description='kcal/day', max=3500.0, min=1200.0, step=50.0), Fl…